## New Ferry Terminal schedule and OPS

"new" in the dataset and variables is the way to call the New Ferry Terminal

In [1]:
from pathlib import Path

import pandas as pd

# Project root (folder that contains this notebook)
ROOT = Path("/Users/michelangeloinze/Desktop/Master_Thesis/coding_MichelangeloInzerilli_MScThesis")
new_OPS_XLSX_PATH = ROOT / "FerryTerminal_coding" / "outputs" / "ferryterminal_loadshifting.xlsx"

df = pd.read_excel(new_OPS_XLSX_PATH)
new_ferryterminal_OPS = df[["datetime", "shore_power_demand_kw"]].copy()

new_ferryterminal_OPS.head()

,datetime,shore_power_demand_kw
0,2025-01-01 00:00:00,0.0
1,2025-01-01 00:15:00,0.0
2,2025-01-01 00:30:00,0.0
3,2025-01-01 00:45:00,0.0
4,2025-01-01 01:00:00,0.0


In [2]:
import pandas as pd

# Project root (folder that contains this notebook)
ROOT = Path("/Users/michelangeloinze/Desktop/Master_Thesis/coding_MichelangeloInzerilli_MScThesis")
new_schedule_XLSX_PATH = ROOT / "FerryTerminal_coding" / "inputs" / "passenger_ship_data_Oskarshamn_2025.xlsx"

# Read Excel and keep only first 3 columns
new_ships_schedule = pd.read_excel(new_schedule_XLSX_PATH).iloc[:, [0, 2, 3]].copy()
new_ships_schedule.columns = ["ship_name", "arrival_dt", "departure_dt"]

# Parse datetime columns
new_ships_schedule["arrival_dt"] = pd.to_datetime(new_ships_schedule["arrival_dt"])
new_ships_schedule["departure_dt"] = pd.to_datetime(new_ships_schedule["departure_dt"])

# Keep schedule as-is (no month/year shifting)
new_ships_schedule = new_ships_schedule.sort_values("arrival_dt").reset_index(drop=True)

In [3]:
new_ships_schedule.tail()

,ship_name,arrival_dt,departure_dt
448,GOTLAND,2025-12-26 19:15:00,2025-12-26 20:10:00
449,GOTLAND,2025-12-27 10:55:00,2025-12-28 11:30:00
450,GOTLAND,2025-12-28 19:15:00,2025-12-28 20:10:00
451,GOTLAND,2025-12-29 20:00:00,2025-12-29 21:05:00
452,VISBY,2025-12-30 20:00:00,2025-12-30 21:05:00


## Existing Ferry Terminal schedule and OPS

After the construction of the New Ferry Terminal, it is supposed that the existing one will only have 30% of the New Ferry Terminal ships arrivals. The decision assumes that the port will unlikely have both of the terminal 100% operationals in terms of ships arrivals. Also the existing base load is decreased, in this case of 30%.

In [4]:
import numpy as np

# 70% reduction in arrivals → keep 30% of rows. Output: existing_ships_schedule (new_ships_schedule unchanged).
KEEP_FRAC = 1.0 - 0.7
rng = np.random.default_rng(42)

prop_before = new_ships_schedule["ship_name"].value_counts(normalize=True)

base = new_ships_schedule.copy()
N = len(base)
K = max(0, int(round(N * KEEP_FRAC)))

counts = base.groupby("ship_name", sort=False).size()
raw = counts.to_numpy(dtype=float) * (K / N)
floored = np.floor(raw).astype(int)
alloc = pd.Series(floored, index=counts.index)
remainder = K - int(alloc.sum())
frac = pd.Series(raw - floored, index=counts.index)

for ship in frac.sort_values(ascending=False).index:
    if remainder <= 0:
        break
    if alloc[ship] < counts[ship]:
        alloc[ship] += 1
        remainder -= 1

base["_pick"] = rng.random(N)
parts = []
for ship, n_take in alloc.items():
    if n_take <= 0:
        continue
    parts.append(base.loc[base["ship_name"] == ship].sort_values("_pick").head(n_take))

existing_ships_schedule = (
    pd.concat(parts, axis=0)
    .drop(columns="_pick")
    .sort_values("arrival_dt", kind="mergesort")
    .reset_index(drop=True)
)

prop_after = existing_ships_schedule["ship_name"].value_counts(normalize=True)

existing_ships_schedule.tail()


,ship_name,arrival_dt,departure_dt
131,GOTLAND,2025-12-19 19:15:00,2025-12-19 20:10:00
132,GOTLAND,2025-12-21 19:15:00,2025-12-21 20:10:00
133,GOTLAND,2025-12-22 20:00:00,2025-12-22 21:05:00
134,GOTLAND,2025-12-26 19:15:00,2025-12-26 20:10:00
135,GOTLAND,2025-12-28 19:15:00,2025-12-28 20:10:00


Ship stays are **redistributed across the analysis year**: each visit keeps the same duration, but both the **calendar date** and **time of day** change (e.g. a stay originally on 03/04 19:15–09/04 08:00 is placed elsewhere in 2025 at a new random 15-minute slot). Stays are placed so that **at most one ship** is at the existing terminal at any time.

In [5]:
import numpy as np
import pandas as pd

SCHEDULE_SHIFT_SEED = 42  # deterministic shuffle for target day and time slots
SCHEDULE_STEP_MINUTES = 15  # minimum gap when resolving overlaps


def shift_stays_across_year(schedule, year_start, year_end, seed=42, step_minutes=15):
    """Move stay intervals to new calendar dates and times of day without overlap.

    Each stay keeps its duration; both date and arrival time-of-day change.
    Target slots use shuffled days and random 15-minute arrival times, then
    are placed greedily to avoid overlap.
    """
    out = schedule.copy()
    out["arrival_dt"] = pd.to_datetime(out["arrival_dt"])
    out["departure_dt"] = pd.to_datetime(out["departure_dt"])

    duration = out["departure_dt"] - out["arrival_dt"]

    year_start = pd.Timestamp(year_start)
    year_end = pd.Timestamp(year_end)
    rng = np.random.default_rng(seed)

    total_days = (year_end.normalize() - year_start.normalize()).days
    n = len(out)
    day_slots = rng.permutation(np.linspace(0, max(total_days - 1, 0), num=n, dtype=int))

    steps_per_day = int(pd.Timedelta(days=1) / pd.Timedelta(minutes=step_minutes))
    time_slot_steps = rng.integers(0, steps_per_day, size=n)
    time_slots = pd.to_timedelta(time_slot_steps * step_minutes, unit="m")

    items = []
    for i in range(n):
        day = year_start.normalize() + pd.Timedelta(days=int(day_slots[i]))
        cand_arr = day + time_slots[i]
        items.append((cand_arr, i))

    items.sort(key=lambda item: item[0])

    occupied = []
    new_arrivals = [None] * n
    new_departures = [None] * n
    step = pd.Timedelta(minutes=step_minutes)

    for cand_arr, i in items:
        arr = cand_arr
        dep = arr + duration.iloc[i]

        while any(arr < occ_end and dep > occ_start for occ_start, occ_end in occupied):
            arr += step
            dep = arr + duration.iloc[i]

        if dep > year_end:
            arr = year_start + (arr - year_start.normalize())
            dep = arr + duration.iloc[i]
            while any(arr < occ_end and dep > occ_start for occ_start, occ_end in occupied):
                arr += step
                dep = arr + duration.iloc[i]

        if dep > year_end:
            raise ValueError(f"Stay index {i} does not fit in the analysis year after shifting.")

        new_arrivals[i] = arr
        new_departures[i] = dep
        occupied.append((arr, dep))

    out["arrival_dt"] = pd.to_datetime(new_arrivals)
    out["departure_dt"] = pd.to_datetime(new_departures)
    return out.sort_values("arrival_dt", kind="mergesort").reset_index(drop=True)


schedule_year_start = pd.to_datetime(df["datetime"]).min()
schedule_year_end = pd.to_datetime(df["datetime"]).max()

existing_ships_schedule_before_shift = existing_ships_schedule.copy()
existing_ships_schedule = shift_stays_across_year(
    existing_ships_schedule_before_shift,
    year_start=schedule_year_start,
    year_end=schedule_year_end,
    seed=SCHEDULE_SHIFT_SEED,
    step_minutes=SCHEDULE_STEP_MINUTES,
)

# Sanity check: no overlapping stays in the shifted schedule
_arrivals = pd.to_datetime(existing_ships_schedule["arrival_dt"])
_departures = pd.to_datetime(existing_ships_schedule["departure_dt"])
_overlap_count = sum(
    1
    for i in range(len(existing_ships_schedule))
    for j in range(i + 1, len(existing_ships_schedule))
    if _arrivals.iloc[i] < _departures.iloc[j] and _arrivals.iloc[j] < _departures.iloc[i]
)
assert _overlap_count == 0, f"Shifted schedule has {_overlap_count} overlapping stay pairs."

existing_ships_schedule.head()

,ship_name,arrival_dt,departure_dt
0,VISBY,2025-01-01 06:15:00,2025-01-01 07:20:00
1,DROTTEN,2025-01-03 11:30:00,2025-01-03 12:40:00
2,VISBY,2025-01-06 08:00:00,2025-01-06 08:55:00
3,DROTTEN,2025-01-09 01:45:00,2025-01-09 02:50:00
4,VISBY,2025-01-11 02:45:00,2025-01-11 03:50:00


In [6]:
# Share of each ship_name (sums to 1 within each dataset)
share_new = new_ships_schedule["ship_name"].value_counts(normalize=True).rename("share_new")
share_existing = existing_ships_schedule["ship_name"].value_counts(normalize=True).rename("share_existing")

ratios = pd.concat([share_new, share_existing], axis=1).fillna(0)
ratios["ratio_existing_over_new"] = ratios["share_existing"] / ratios["share_new"].replace(0, float("nan"))

ratios.sort_index()

,share_new,share_existing,ratio_existing_over_new
ship_name,,,
DROTTEN,0.532009,0.529412,0.995118
GOTLAND,0.225166,0.227941,1.012327
VISBY,0.242826,0.242647,0.999265


Now calculating existing terminal OPS demand

In [7]:
import numpy as np
import pandas as pd

# Expected input table: one row per ferry stay with:
# - arrival_dt
# - departure_dt
# - ship_name
# Example source variable: existing_ships_schedule

# 1) Prepare intervals
ships_intervals = existing_ships_schedule[["arrival_dt", "departure_dt", "ship_name"]].copy()
ships_intervals["arrival_dt"] = pd.to_datetime(ships_intervals["arrival_dt"])
ships_intervals["departure_dt"] = pd.to_datetime(ships_intervals["departure_dt"])

# Power by ship (kW)
power_map_kw = {
    "VISBY": 1492,
    "GOTLAND": 1494,
    "DROTTEN": 1368,
}
ships_intervals["power_kw"] = ships_intervals["ship_name"].astype(str).str.upper().map(power_map_kw).fillna(900.0)

# 1b) Snap to 15-minute steps (00, 15, 30, 45)
# - arrivals -> previous step (floor)
# - departures -> next step (ceil)
ships_intervals["arrival_dt"] = ships_intervals["arrival_dt"].dt.floor("15min")
ships_intervals["departure_dt"] = ships_intervals["departure_dt"].dt.ceil("15min")

# Optional: drop invalid intervals
ships_intervals = ships_intervals[ships_intervals["departure_dt"] > ships_intervals["arrival_dt"]]

# 2) Define 15-minute window from data
start_step = ships_intervals["arrival_dt"].min().floor("15min")
end_step = ships_intervals["departure_dt"].max().ceil("15min")
step_edges = pd.date_range(start=start_step, end=end_step, freq="15min")

# 3) Build 15-minute occupancy matrix
step_index = step_edges[:-1]
start_col = ships_intervals["arrival_dt"].to_numpy()[:, None]
end_col = ships_intervals["departure_dt"].to_numpy()[:, None]
mask = (step_index.to_numpy()[None, :] >= start_col) & (step_index.to_numpy()[None, :] < end_col)

# Per-ship power multipliers by position in stay:
# - if stay <= 4h: first 15 min = 0.85, last 15 min = 0.85
# - if stay  > 4h:
#   - first 15 min after arrival: 0.425
#   - second 15 min after arrival: 0.85
#   - second-last 15 min before departure: 0.85
#   - last 15 min before departure: 0.425
factors = mask.astype(float)
max_steps_short_stay = int(pd.Timedelta(hours=4) / pd.Timedelta(minutes=15))  # 16 steps

for i in range(factors.shape[0]):
    active_idx = np.flatnonzero(mask[i])
    if active_idx.size == 0:
        continue

    if active_idx.size <= max_steps_short_stay:
        # Short stay profile
        factors[i, active_idx[0]] = 0.85
        factors[i, active_idx[-1]] = 0.85
        continue

    # Long stay profile
    # Arrival ramp
    factors[i, active_idx[0]] = 0.425
    factors[i, active_idx[1]] = 0.85

    # Departure ramp
    factors[i, active_idx[-1]] = 0.425
    factors[i, active_idx[-2]] = 0.85

# Demand per step = sum(power_kW * factor of active ships)
step_energy_kwh = (factors * ships_intervals["power_kw"].to_numpy()[:, None]).sum(axis=0)

# 4) Final dataset: existing_ships_demand
existing_ships_demand = pd.DataFrame({"datetime": step_index, "demand_kW": step_energy_kwh})

existing_ships_demand.head()

,datetime,demand_kW
0,2025-01-01 06:15:00,1268.2
1,2025-01-01 06:30:00,1492.0
2,2025-01-01 06:45:00,1492.0
3,2025-01-01 07:00:00,1492.0
4,2025-01-01 07:15:00,1268.2


Now scaling down the base demand

In [8]:
BASE_DEMAND_SCALE = 0.3  # existing terminal base load at 30% of old profile

existing_ferryterminal_basedemand = df[["datetime", "base_demand_kw"]].copy()
existing_ferryterminal_basedemand["base_demand_kw"] *= BASE_DEMAND_SCALE

existing_ferryterminal_basedemand.head()

,datetime,base_demand_kw
0,2025-01-01 00:00:00,54.648063
1,2025-01-01 00:15:00,54.602636
2,2025-01-01 00:30:00,55.041759
3,2025-01-01 00:45:00,54.860053
4,2025-01-01 01:00:00,54.118087


In [9]:
existing_ferryterminal_demand = existing_ships_demand.merge(
    existing_ferryterminal_basedemand,
    on="datetime",
    how="outer",
    sort=True,
)
existing_ferryterminal_demand[["demand_kW", "base_demand_kw"]] = existing_ferryterminal_demand[
    ["demand_kW", "base_demand_kw"]
].fillna(0.0)
existing_ferryterminal_demand["total_kW"] = (
    existing_ferryterminal_demand["demand_kW"] + existing_ferryterminal_demand["base_demand_kw"]
)

existing_ferryterminal_demand.head()

,datetime,demand_kW,base_demand_kw,total_kW
0,2025-01-01 00:00:00,0.0,54.648063,54.648063
1,2025-01-01 00:15:00,0.0,54.602636,54.602636
2,2025-01-01 00:30:00,0.0,55.041759,55.041759
3,2025-01-01 00:45:00,0.0,54.860053,54.860053
4,2025-01-01 01:00:00,0.0,54.118087,54.118087


In [10]:
out = Path("inputs") / "existing_ferryterminal_demand.xlsx"
existing_ferryterminal_demand.to_excel(out, index=False)